# Knowledge Extractor - Extracción de Entidades ATC

Este notebook procesa documentos PDF de fraseología aeronáutica para extraer entidades y relaciones usando NLP con Ollama.

## Importación de librerías

Importamos todas las dependencias necesarias para el procesamiento:
- `ollama`: Para interactuar con el modelo de lenguaje local
- `json`: Para manejar estructuras de datos JSON
- `re`: Para expresiones regulares
- `os`: Para operaciones de sistema de archivos
- `pypdf`: Para leer y extraer texto de archivos PDF

In [1]:
import ollama
import json
import re
import os
from pypdf import PdfReader

## Función NER con Ollama y Contexto

Esta función utiliza el modelo de lenguaje Ollama para realizar Named Entity Recognition (NER) y extracción de relaciones en texto de control de tráfico aéreo, **ahora con contexto de entidades previas**.

**Características:**
- Extrae entidades aeronáuticas (aeródromos, altitudes, aeronaves, etc.)
- Identifica relaciones operativas entre entidades
- Usa un prompt especializado para dominio ATC
- **NUEVO: Mantiene consistencia con entidades extraídas en páginas anteriores**
- Devuelve resultados en formato JSON estructurado

**Parámetros:**
- `texto`: Texto de entrada para analizar
- `entidades_previas`: Lista de entidades extraídas en páginas anteriores (opcional)
- `modelo`: Modelo de Ollama a usar (default: "llama3")

**Mejoras implementadas:**
- Contexto de entidades previas para mantener consistencia
- Nuevo constraint que exige usar texto y etiquetas idénticos para entidades recurrentes
- Deduplicación automática de entidades acumuladas

In [2]:
def ner_con_ollama(texto, entidades_previas=None, modelo="llama3"):
    """Usa Ollama para NER con contexto de entidades previas"""

    prompt = """
        ATC NER & Relationship Extraction Prompt
        "Extract ALL instances of aeronautical entities and their operational relationships in the following JSON format:

        {
        "entities": [
            {
            "text": "the exact text from the source",
            "label": "the category of the entity",
            "context": "brief semantic description"
            }
        ],
        "relationships": [
            {
            "subject": "Entity A",
            "predicate": "The relationship or action connecting them",
            "object": "Entity B"
            }
        ]
        }

        [Specify Constraints]:

            Focus: Only consider entities and relationships related to Air Traffic Control (ATC) procedures, flight constraints (altitude, speed, heading, position), and facility responsibilities.
            Dynamic Discovery: Do not limit extraction to a fixed list. While you should prioritize standard categories (such as Aerodrome, Altitude, ARTCC, Fix/Position, Sector, and Tower), you must also extract any other domain-specific entity discovered in the text that impacts flight operations or airspace management.
            Relationship Type: Prioritize extracting "subject-predicate-object" triplets that define operational rules or constraints (e.g., "Arrivals - must use - CEDES STAR").
            Entity Consistency: Maintain strict consistency with previously extracted entities. Use identical text and labels for recurring entities. If you encounter an entity that matches a previously extracted one, use the exact same text and label rather than creating variations.
    """

    # Añadir entidades previas si existen
    if entidades_previas:
        prompt += "\n\n[Previously Extracted Entities]:\n"
        for entidad in entidades_previas:
            prompt += f"- {entidad['text']} ({entidad['label']})\n"
    
    prompt += "\n\n[Input Specification]:\n"
    prompt += texto + "\nJSON Output ONLY:\n"

    response = ollama.chat(model=modelo, messages=[
        {
            'role': 'user',
            'content': prompt 
        },
    ])

    return response['message']['content']

## Extractor de JSON robusto

Esta función extrae JSON válido de texto que puede contener bloques de código o respuestas de LLM.

**Problema resuelto:** JSON no es un lenguaje regular, no se puede parsear con regex debido a anidación infinita.

**Solución:** Parser manual que:
- Busca balance correcto de llaves `{}` considerando strings y escapes
- Maneja bloques de código ```json``` o ```
- Ignora llaves dentro de strings
- Procesa caracteres de escape correctamente
- Soporta anidación arbitraria

**Ventajas sobre regex:**
- Maneja estructuras JSON complejas
- Evita falsos positivos por llaves en strings
- Soporta JSON multilinea correctamente

In [3]:
def extract_json(texto):
    """
    Extrae JSON de un texto usando un parser manual que maneja anidación correctamente.
    """
    
    def encontrar_json_por_llaves(texto):
        """
        Encuentra el JSON válido buscando el balance correcto de llaves {}
        """
        for i, char in enumerate(texto):
            if char == '{':
                # Encontrar el cierre de llave correspondiente
                balance = 1
                j = i + 1
                escape_next = False
                in_string = False
                string_char = None
                
                while j < len(texto) and balance > 0:
                    char_actual = texto[j]
                    
                    if not escape_next:
                        if char_actual == '\\' and in_string:
                            escape_next = True
                        elif char_actual in ['"', "'"] and not escape_next:
                            if not in_string:
                                in_string = True
                                string_char = char_actual
                            elif char_actual == string_char:
                                in_string = False
                                string_char = None
                        elif not in_string:
                            if char_actual == '{':
                                balance += 1
                            elif char_actual == '}':
                                balance -= 1
                    
                    if escape_next:
                        escape_next = False
                    
                    j += 1
                
                if balance == 0:
                    # Encontramos un JSON potencial
                    json_str = texto[i:j]
                    try:
                        return json.loads(json_str), j
                    except json.JSONDecodeError:
                        continue
        
        return None, -1
    
    # Primero intentar encontrar y eliminar bloques de código ```json o ```
    texto_limpio = texto
    patron_bloque = r'```(?:json)?\s*\n(.*?)\s*```'
    match_bloque = re.search(patron_bloque, texto, re.DOTALL)
    
    if match_bloque:
        # Extraer el contenido del bloque
        contenido_bloque = match_bloque.group(1).strip()
        resultado, pos = encontrar_json_por_llaves(contenido_bloque)
        if resultado:
            return resultado
    
    # Si no funciona con bloque, buscar en todo el texto
    resultado, pos = encontrar_json_por_llaves(texto_limpio)
    if resultado:
        return resultado
    
    return None


## Configuración inicial

Definimos las rutas y parámetros para el procesamiento:
- `doc_path`: Ruta al archivo PDF de entrada
- `output_dir`: Directorio base para archivos de salida
- `doc_name`: Nombre del documento derivado del filename

Esta configuración permite procesar diferentes documentos cambiando solo las variables iniciales.

In [ ]:
def configure_output(doc_path, output_dir, model_name):
    # Configuración
    doc_name = doc_path.split("/")[-1].split(".")[0] + f"({model_name})"

    # Crear directorio de salida si no existe
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Creado directorio: {output_dir}")

    # Crear carpeta para el documento si no existe
    doc_dir = os.path.join(output_dir, doc_name)
    if not os.path.exists(doc_dir):
        os.makedirs(doc_dir)
        print(f"Creada carpeta para el documento: {doc_dir}")

    return doc_dir

## Procesamiento principal del PDF con Contexto Acumulado

Esta celda ejecuta el flujo completo de procesamiento **mejorado con contexto**:

1. **Extracción de texto**: Lee cada página del PDF y extrae el texto
2. **NER con Ollama**: Aplica reconocimiento de entidades usando el modelo LLM con contexto de entidades previas
3. **Acumulación de entidades**: Mantiene una lista creciente de entidades únicas encontradas
4. **Deduplicación**: Evita repetir entidades idénticas basadas en texto normalizado
5. **Parseo de JSON**: Extrae el JSON estructurado de la respuesta del LLM
6. **Guardado**: Crea archivos JSON por página con:
   - `texto_original`: Texto extraído del PDF
   - `ner`: Entidades y relaciones extraídas (o texto plano si falla)
   - `contexto_entidades_usadas`: Número de entidades de contexto disponibles

**Mejoras clave:**
- **Consistencia**: El LLM recibe contexto de entidades previas para mantener nomenclatura consistente
- **Deduplicación**: Sistema automático para evitar entidades duplicadas
- **Métricas**: Registro de cuántas entidades de contexto se usaron por página

**Salida:** Archivos `pagina_N.json` en el directorio del documento con información de contexto

**Manejo de errores:** Si el parseo JSON falla, guarda la respuesta original del LLM como fallback.

In [5]:
def kex(doc_path, doc_dir, model_name):

    # Cargar el documento PDF
    reader = PdfReader(doc_path)
    numero_paginas = len(reader.pages)

    print(f"Procesando {numero_paginas} páginas...")

    # Lista para acumular entidades de páginas anteriores
    entidades_acumuladas = []

    # Procesar cada página y crear JSON
    for i, pagina in enumerate(reader.pages):
        # Extraer texto de la página
        texto_original = pagina.extract_text()
        
        # Aplicar NER usando Ollama con contexto de entidades previas
        resultado_ner = ner_con_ollama(texto_original, entidades_acumuladas, model_name)
        
        # Extraer JSON del resultado
        ner_json = extract_json(resultado_ner)
        
        if ner_json:
            print(f"Pagina {i+1}: JSON extraído correctamente")
            
            # Acumular nuevas entidades (evitando duplicados)
            if 'entities' in ner_json:
                # Normalizar y deduplicar entidades
                for nueva_entidad in ner_json['entities']:
                    texto_normalizado = nueva_entidad['text'].lower().strip()
                    
                    # Verificar si ya existe una entidad similar
                    existe_similar = False
                    for existente in entidades_acumuladas:
                        if existente['text'].lower().strip() == texto_normalizado:
                            existe_similar = True
                            break
                    
                    if not existe_similar:
                        entidades_acumuladas.append(nueva_entidad)
                
            print(f"  - Entidades acumuladas: {len(entidades_acumuladas)}")
        else:
            print(f"Pagina {i+1}: No se pudo extraer JSON")

        # Crear diccionario con los datos
        pagina_data = {
            "texto_original": texto_original,
            "ner": ner_json if ner_json else resultado_ner,  # Si falla JSON, guardar texto plano
            "contexto_entidades_usadas": len(entidades_acumuladas) if i > 0 else 0
        }
        
        # Guardar como archivo JSON
        filename = f"pagina_{i+1}.json"
        filepath = os.path.join(doc_dir, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(pagina_data, f, ensure_ascii=False, indent=2)
        
        print(f"Creado archivo: {filename}")

    print(f"\nProceso completado. Archivos guardados en: {doc_dir}")
    print(f"Total de entidades únicas acumuladas: {len(entidades_acumuladas)}")

In [ ]:
# Configuración
doc_path = "../docs/ICAO Standard Phraseology.pdf"
output_dir = "kex_output"

doc_dir = configure_output(doc_path, output_dir, "llama3.1")
kex(doc_path, doc_dir, "llama3.1")

---
---

## Reprocesamiento de archivos existentes

Esta celda actualiza los archivos JSON existentes para mejorar la extracción del NER.

**Propósito:**
- Lee archivos JSON ya generados que contienen `llm_output` crudo
- Aplica la función `extract_json()` mejorada para parsear correctamente el NER
- Reemplaza el campo `ner` con el JSON estructurado

**Manejo de errores robusto:**
- Verifica archivos vacíos antes de procesar
- Captura `JSONDecodeError` para archivos corruptos
- Maneja excepciones generales
- Reporta estado de cada archivo procesado

**Resultado:** Archivos JSON con campo `ner` correctamente estructurado en lugar de texto plano.

In [11]:
from pathlib import Path

p = Path("kex_output/ICAO Standard Phraseology")

for file in p.iterdir():
    file_path = "/".join(file.parts)
    
    try:
        # Leer el archivo primero
        with open(file_path, 'r', encoding='utf-8') as f:
            pag_lines = "".join(f.readlines())
            
            # Verificar que el archivo no esté vacío
            if not pag_lines.strip():
                print(f"Archivo vacío: {file.name}")
                continue
                
            pag_json = json.loads(pag_lines)
        
        # Procesar el NER solo si existe llm_output
        if "llm_output" in pag_json:
            pag_json["ner"] = extract_json(pag_json["llm_output"])
        else:
            print(f"No hay llm_output en: {file.name}")
        
        # Escribir el archivo actualizado
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(json.dumps(pag_json, indent=4))
        
        print(f"Procesado: {file.name}")
        
    except json.JSONDecodeError as e:
        print(f"Error JSON en {file.name}: {e}")
    except Exception as e:
        print(f"Error procesando {file.name}: {e}")

Archivo vacío: pagina_6.json
Procesado: pagina_5.json
Procesado: pagina_11.json
Procesado: pagina_12.json
Procesado: pagina_3.json
Procesado: pagina_16.json
Procesado: pagina_17.json
Procesado: pagina_13.json
Procesado: pagina_14.json
Procesado: pagina_8.json
Procesado: pagina_2.json
Procesado: pagina_4.json
Procesado: pagina_15.json
Procesado: pagina_1.json
Procesado: pagina_20.json
Procesado: pagina_10.json
Procesado: pagina_19.json
Procesado: pagina_7.json
Procesado: pagina_18.json
Procesado: pagina_9.json


In [18]:
from pathlib import Path
import pandas as pd
import json

df = pd.DataFrame(columns=["model"] + ["pagina_" + str(i) for i in range(1,21)])
p = Path("kex_output")

for folder in p.iterdir():
    ner_pages = {}
    for file in folder.iterdir():
        key = int(file.stem.split("_")[-1])
        file_path = "/".join(file.parts)
        try:
            # Leer el archivo primero
            with open(file_path, 'r', encoding='utf-8') as f:
                pag_lines = "".join(f.readlines())
                
                # Verificar que el archivo no esté vacío
                if not pag_lines.strip():
                    print(f"Archivo vacío: {file.name}")
                    ner_pages[key] = ""
                    continue
                    
                pag_json = json.loads(pag_lines)
            
            ner_pages[key] = pag_json["ner"]    
        except json.JSONDecodeError as e:
            print(f"Error JSON en {file.name}: {e}")
        except Exception as e:
            print(f"Error procesando {file.name}: {e}")

    ner_pages_sort = []
    for i in sorted(ner_pages):
        ner_pages_sort.append(ner_pages[i])
    
    df.loc[len(df)] = [folder.stem] + ner_pages_sort

Archivo vacío: pagina_6.json
